# 8j — Composable forecast (four ways ++ two baselines), TWO-STAGE cut

Implements `inst/1a_preliminary_framework_plan.md` + `inst/1c`/`inst/1d`/`inst/1e`, and the
**two-stage cut inference** of `inst/4_cut_Bayes.md`: a renewal / next-generation-matrix model
driven by **age-pair contact-degree distributions**, scored **four ways** — the 2×2 grid of
{unweighted **NegBin**, weighted **Hurdle-Weibull**} degree models × {**Mean**, **Neighbourhood**} NGM —
plus the **two baselines** of `inst/6_null_interaction_model.md`:

- **no-interaction** (`unweighted-negbin|mean-diagonal`) — NegBin degree, mean NGM with only the
  **diagonal** of C* kept, so age groups do not infect each other; age-dependent infectivity is
  pinned to `inf ≡ 1` for identifiability. It **reuses the NegBin Stage-1 chains verbatim**
  (`8j_s1_*` carry no ngm token), so it adds no Stage-1 fits.
- **null** (`no-contact|null`) — **no social contact data at all**: C* is a fixed uniform constant,
  the average unweighted number of contacts over the origin's 8 focal weeks, held fixed across
  horizons. Stage 1 is skipped entirely; only the transmission parameters (age-dependent
  susceptibility/infectivity, generation interval) drive the forecast. Its purpose is to say what
  the contact data actually buys.

**Cut inference.** The former joint fit is split: **Stage 1** fits the contact-degree GP alone
(`model_degree`), and **Stage 2** fits the infection/renewal block (`model_transmission`)
conditioning on a *fixed* contact matrix drawn from Stage 1. Stage-1 uncertainty is propagated by
imputing **100** Stage-1 posterior draws, re-fitting Stage 2 for each (keeping **100** draws), and
**pooling** the 100×100 = **10 000** infection draws as the predictive used for WIS.

The contact **mean** is estimated **per week** with **structural reciprocity**
(`log μ_{i→j} = r + log Nⱼ`) and **separable spatio-temporal-GP smoothing** across the age-pair
grid *and over weeks* (inst/1e, §5). The NGM uses the **per-contact secondary attack rate γ_SAR**
with **un-normalised** C* (the `-gnorm` C*/S̄ decoupling was reverted). Forecasts use the
**contact-updated iterate** over origins × 4 horizons; **WIS** is on a **log scale**, by horizon.

**Sampling (2026-08-05).** Stage 1 uses **Turing NUTS** (`STAGE1_USE_NUTS = true`, now the default),
initialised from the **Pathfinder mean** — Pathfinder still runs first, so the cost is *additive*.
**Stage 2 is always Pathfinder** and has no NUTS path: 100 cheap fits per Stage-1 draw is the point
of the cut. Gradients for both stages come from **Mooncake** (`AD_BACKEND = :mooncake`), ~9–18×
faster than the previous ReverseDiff backend at identical gradients.

> ⚠ Switching Stage 1 to NUTS starts a **new cache generation** (`…-gi-nuts`). The existing 504/1512
> Pathfinder-generation files are untouched and remain reachable as `CONTACTS_TOKEN_PF` — but until
> this notebook has been re-run to completion, `9j`/`10j`/`11j` will find no `-nuts` artefacts.

> **This notebook does FITTING ONLY.** It fits and caches both stages' artefacts (the slow part).
> Forecast assembly, WIS scoring, and diagnostics moved to `9j_forecast_diagnostics.ipynb`, which
> reloads these files.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

# Stage-1 (contact-degree GP) sampler. true ⇒ Turing NUTS — THE DEFAULT since 2026-08-05 (user
# request: "use only NUTS in stage 1"). NUTS is initialised from the Pathfinder mean
# (`_pf_mean_init`), so Pathfinder still runs first and the cost is ADDITIVE, not a replacement.
# false ⇒ Pathfinder only, the preliminary generation that produced the existing 504-file grid.
# STAGE 2 (infection) IS ALWAYS PATHFINDER and has no NUTS path at all — 100 cheap fits per Stage-1
# draw is the point of the cut (inst/4_cut_Bayes.md); that is by design, not a gap.
#
# ⚠ This flag IS in the cache-filename token (`contacts_label` appends `-nuts`), so switching it
# does NOT reuse the Pathfinder chains — it starts a new generation. The old one stays on disk and
# is reachable as `CONTACTS_TOKEN_PF`.
STAGE1_USE_NUTS = true

# AD backend for BOTH stages' gradients. :mooncake is the default (source-to-source, no tracked
# element types); :reversediff is the fallback the previous generation was fitted with.
# Measured 2026-08-05 at origin 2021-05-09, Mooncake vs ReverseDiff gradients/s:
#   Stage-1 negbin (389 dims)  482 vs 44   (10.9×)
#   Stage-1 hweibull (977)     241 vs 27   ( 9.0×)
#   Stage-2 transmission (18)  30685 vs 1711 (17.9×)
# with gradients agreeing to ≤4e-14 relative. Mooncake pays a one-off rule build per model type per
# process (~66 s / 14 s / 15 s), which `prefit_stage1!` warms serially before its thread fan-out.
# ⚠ NOT in the cache token — flipping it silently reuses chains. It is recorded INSIDE each
# `8j_s1_*` file under the `ad_backend` key instead.
AD_BACKEND = :mooncake

## §1 Window, infection/antibody data, and age-pair degree data

In [ ]:
# `constant_contacts = false` ⇒ contact degree estimated PER WEEK, temporally smoothed by a
# separable spatio-temporal GP (shared ρ_diag/ρ_time, η, σ_c; scalar intercept c +
# temporal-level GP cₜ = c + σ_c·(Lt·z_c) + sum-to-zero matrix-normal field η·(Q·La·z·Ltᵀ)). The renewal NGM
# then varies in time through contacts as well as antibody: N(t) uses that week's C*ₜ.
# (Set true for the pooled one-C*-per-window preliminary.)
# `stage1_use_nuts` selects the Stage-1 (contact-degree) sampler; Stage 2 is always Pathfinder.
# `ad_backend` selects the AD backend for BOTH stages' gradients (see cell 1).
cfg  = FrameworkConfig(constant_contacts = false, stage1_use_nuts = STAGE1_USE_NUTS,
                       ad_backend = AD_BACKEND)
grid = cis_age_grid()

# Read the CoMix contact data AND the inc2prev infection/antibody estimates ONCE and reuse them
# across every window (avoids re-reading/re-joining the full Arrow and re-parsing the estimates
# CSV per origin×horizon). Then roll the forecast origin over the whole period the current
# datasets support ("available period"): each origin needs a 12-week fit/lag window back to the
# first inc2prev week, and contact data out to origin+4 for the contact-updated iterate
# (horizon-h contacts observed at t₀+h). `available_forecast_origins` derives the range.
#
# `FIT_END` caps the roll at the last week STARTING in 2021 (⇒ last origin 2021-12-26; its h=1..4
# targets still run into Jan 2022, well inside both datasets). The data-derived bound alone is far
# too generous: CoMix's main panel stops 2022-03-02, but a stray 2022-11-16…28 block pushes it out
# to 2022-10-30, so origins from ~2022-03-06 would roll through a window with no contact data at
# all. inc2prev is a second, uncaught limit — infections/antibody end 2022-03-26 and later weeks
# are silently zero-filled rather than erroring.
FIT_END = Date(2021, 12, 31)     # snapped to the Sunday-start grid by week_start ⇒ 2021-12-26
raw  = load_raw_contact_inputs()
inf  = load_raw_infection_inputs()          # (; df, tmap) — read estimates_age_ab.csv once
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw,
                                             origin_max = FIT_END)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

println("contact data span  : ", extrema(skipmissing(raw.craw.date)))
println("forecast origins   : ", length(wins), " weekly, ",
        first(FORECAST_ORIGINS), " … ", last(FORECAST_ORIGINS), " (capped at ", FIT_END, ")")
println("sampler / AD       : Stage 1 ", cfg.stage1_use_nuts ? "NUTS" : "Pathfinder",
        ", Stage 2 Pathfinder | ad_backend = ", cfg.ad_backend,
        " | cache token = ", contacts_label(cfg))
let w = wins[1], wd0 = load_window_data(wins[1], inf.df, inf.tmap; grid = grid)
    println("origin[1] fit weeks: ", w.fit_weeks[1], " … ", w.fit_weeks[end])
    println("weekly infections @ origin[1] (age): ", round.(wd0.I_mean[:, end]; digits = 0))
end

In [ ]:
# Fit config: the SIX model variants and the parallel-fit concurrency (CPU- and memory-balanced).
#   • the "four ways" 2×2 grid: {unweighted NegBin, weighted Hurdle-Weibull} × {mean, neighbourhood}
#   • no-interaction (inst/6): NegBin degree, MEAN NGM with only the DIAGONAL of C* kept, and
#     age-dependent infectivity pinned to 1 (identifiability). It reuses the NegBin Stage-1 chains
#     verbatim (`8j_s1_*` carry no ngm token) ⇒ no extra Stage-1 fits.
#   • null (inst/6): no contact data at all — C* is a fixed uniform constant, the average unweighted
#     number of contacts over the origin's 8 focal weeks, held fixed across horizons. Skips Stage 1
#     entirely; one Stage-2 fit per (origin × horizon) with the full 10 000 draws.
combos = vcat([(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                        for nb in (MeanNGM(), NeighbourhoodDegreeNGM())],
              [(NegBinAgePair(),   DiagonalMeanNGM()),      # no-interaction
               (NoContactDegree(), NullNGM())])             # null
MAX_FIT_CONCURRENCY = fit_concurrency()          # min(threads, cores−1, RAM-budget)

# Print the FULL breakdown, not just the number: this value decides whether the pre-fit below is
# serial or n-way, and it used to collapse to 1 SILENTLY on macOS (`_mem_available_gib` read
# /proc/meminfo, which does not exist there, and fell back to `Sys.free_memory()` = truly-free
# pages only — ~2 GiB on an idle 32 GiB machine — so the memory cap floored to 0). Fixed
# 2026-07-30 via a `vm_stat` darwin branch; `binding` now says which cap actually binds.
let r = fit_concurrency_report()
    println("fit concurrency = ", r.concurrency, "  (binding: ", r.binding, ")")
    println("  memory: ", round(r.avail_gib, digits = 1), " GiB available ⇒ mem_cap = ", r.mem_cap)
    println("  cpu   : nthreads = ", r.nthreads, ", Sys.CPU_THREADS = ", r.cpu_threads,
            " ⇒ cpu_cap = ", r.cpu_cap)
end
if Threads.nthreads() == 1
    @warn "Julia has 1 thread — pre-fit runs sequentially. Start with JULIA_NUM_THREADS>1 " *
          "(e.g. $(max(1, Sys.CPU_THREADS - 1))) for parallel fitting."
end
println("combos = ", length(combos), " | fit concurrency = ", MAX_FIT_CONCURRENCY,
        " | total fits = ", length(wins) * length(combos) * length(cfg.horizons),
        " (cached ones are skipped)")

## §2 Roll over the available period — two-stage fit, forecast 1–4 weeks ahead

For **each weekly origin** across the available period the six model variants are fit in two stages
and forecast. The contact **mean** is a **reciprocity-structural, GP-smoothed** field
(`log μ_{i→j}=r+log Nⱼ` ⟹ exact reciprocity), estimated per week over the fit window **and the
forecast weeks** — the horizon-`h` contact window ends at `t₀+h`, separately per horizon (inst/1d).

**Stage 1 (`prefit_stage1!`)** fits that GP once per (degree × origin × horizon) — it is
NGM-independent, so one fit serves every builder of that degree family (including the
no-interaction model). The **null** model has no Stage 1 at all and is skipped here. **Stage 2 (`prefit_stage2!`)** then, per
(degree × ngm × origin × horizon), imputes 100 Stage-1 posterior draws, re-fits the infection block
`model_transmission` conditioning on each (Pathfinder), and pools 100×100 = 10 000 infection draws.
Forecasting is the **contact-updated iterate**: per origin t₀ and horizon `h` the NGM uses the
Stage-1 draw's `t₀+h` C* and the paired Stage-2 infection draw, and one renewal step is taken.

Origins run **sequentially** (bounded memory); within an origin Stage-1 fits fan out over threads
and each Stage-2 cell fans out its 100 per-draw fits (`MAX_FIT_CONCURRENCY`). Both stages are
cached (`8j_s1_*` / `8j_s2_*` under `../dt_intermediate`), so the run is **resumable** — a re-run
reloads finished files and only fits what's missing.

In [ ]:
# Parallel, resumable TWO-STAGE PRE-FIT — the cut inference (inst/4_cut_Bayes.md).
#   Stage 1: fit the contact-degree GP once per (degree × origin × horizon) — NGM-independent —
#            and cache it to ../dt_intermediate/8j_s1_<degree>_<contacts>_<origin>_h<h>.jld2.
#   Stage 2: for each (degree × ngm × origin × horizon), impute 100 Stage-1 posterior draws, re-fit
#            the infection block conditioning on each (Pathfinder), keep 100 draws, and cache the
#            100×100 = 10_000 pooled infection draws to 8j_s2_<degree>_<ngm>_<contacts>_<origin>_h<h>.jld2.
# Origins run sequentially (bounded memory); within an origin Stage-1 fits fan out over threads and
# each Stage-2 cell fans out its 100 per-draw fits. Cached files are skipped ⇒ resumable. Forecast
# assembly, WIS scoring, and diagnostics live in 9j_forecast_diagnostics.ipynb (run this first).

# One origin's datasets: window infection/antibody + the 4 contact/degree windows. Pure &
# deterministic given the shared read-only `inf`/`raw` reads, so it is safe to call from the
# per-origin prefit driver.
build_origin_data(oi, win_o) = (
    load_window_data(win_o, inf.df, inf.tmap; grid = grid),                     # reuse the single CSV read
    [prepare_degree_data(
         WeeklyWindow(win_o.origin + Day(7 * h);
                      n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons),
         cfg; grid = grid, setting = :all,
         df_part_raw = raw.df_part, craw_raw = raw.craw)                        # reuse the single Arrow read
     for h in cfg.horizons])

t0  = time()
res = prefit_two_stage!(combos, wins, cfg;
          data_provider = build_origin_data,
          save_dir = "../dt_intermediate", max_concurrent = MAX_FIT_CONCURRENCY)
println("two-stage pre-fit: Stage 1 ", res.stage1.fitted, " fitted / ", res.stage1.skipped,
        " skipped; Stage 2 ", res.stage2.fitted, " fitted / ", res.stage2.skipped,
        " skipped in ", round(Int, time() - t0), "s")

# Verifiable tail: count how many of this run's Stage-2 pooled files are present.
s2_paths = ["../dt_intermediate/8j_s2_$(degree_label(dm))_$(ngm_label(nb))_" *
            "$(contacts_label(cfg))_$(win.origin)_h$(h).jld2"
            for win in wins, (dm, nb) in combos, h in cfg.horizons]
n_have, n_expect = count(isfile, s2_paths), length(s2_paths)
println("pooled Stage-2 files: $n_have / $n_expect present under ../dt_intermediate ",
        "(", contacts_label(cfg), " contacts) — feed 9j_forecast_diagnostics.ipynb")